# Quran Muaalem dataset loading

This is a small, read-only streaming demonstration. It fetches only a few records and decodes a single audio clip on demand. From the repository root, install with `python -m pip install -e ".[dev]"` and run with `jupyter nbconvert --to notebook --execute notebooks/02_dataset_loading.ipynb --inplace`.


## 1. Load one configuration and a few records
The configuration defaults to streaming and a three-row sample limit. Each original record is left intact.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / 'src' / 'quran_dataset').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from quran_dataset import DatasetConfig, DatasetLoader, decode_audio, inspect_record

config = DatasetConfig.from_file(ROOT / 'configs' / 'dataset.json')
loader = DatasetLoader(config)
print('Dataset:', config.dataset_id)
print('Selected configurations:', config.selected_configurations)
print('Streaming:', config.streaming, '| limit:', config.sample_limit)

records = list(loader.iter_samples('moshaf_0.1', split='train', limit=3))
print('Records loaded:', len(records))

Dataset: obadx/muaalem-annotated-v3
Selected configurations: ('moshaf_0.1',)
Streaming: True | limit: 3


Records loaded: 3


## 2. Available fields and compact inspection
The summary omits encoded audio bytes and waveform arrays.

In [2]:
sample = records[0]
summary = inspect_record(sample)
print('Fields:', summary['fields'])
print('Candidate compound ID:', summary['candidate_id'])
print('Text:', sample.get('imlaey'))
print('Phonemes:', sample.get('phonemes'))
print('Sifat count:', summary['sifat_count'])
print('Has Quran:', sample.get('has_quran'))

Fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']
Candidate compound ID: ('0.1', '001.0000')
Text: أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ
Phonemes: ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم
Sifat count: 19
Has Quran: False


## 3. Decode audio without changing it
The dataset is opened with decoding disabled, so this cell decodes only the selected row with `soundfile`. No resampling or other audio processing is applied.

In [3]:
audio = decode_audio(sample)
print('Shape:', audio.samples.shape)
print('Sampling rate:', audio.sampling_rate)
print('Channels:', audio.channels)
print('Duration (s):', audio.duration_seconds)
print('dtype:', audio.samples.dtype)

Shape: (96960, 1)
Sampling rate: 16000
Channels: 1
Duration (s): 6.06
dtype: float32


## 4. Join recording and reciter metadata
Metadata rows are small and loaded separately. The join keeps the original row mapping unchanged.

In [4]:
metadata = loader.metadata_index()
joined = metadata.join(sample)
print('Original moshaf_id:', joined.record.get('moshaf_id'))
print('Moshaf metadata:', {k: joined.moshaf_metadata.get(k) for k in ['id', 'name', 'recitation_speed']} if joined.moshaf_metadata else None)
print('Original reciter_id:', joined.record.get('reciter_id'))
print('Reciter metadata:', {k: joined.reciter_metadata.get(k) for k in ['id', 'english_name', 'country_code']} if joined.reciter_metadata else None)
print('Record fields unchanged:', list(joined.record) == list(sample))

Original moshaf_id: 0.1
Moshaf metadata: {'id': '0.1', 'name': 'المصحف المرتل بمد المنفصل', 'recitation_speed': 'above_murattal'}
Original reciter_id: 0
Reciter metadata: {'id': 0, 'english_name': 'Mahmoud Khalil Al-Husari', 'country_code': 'EG'}
Record fields unchanged: True


## 5. Inspect schemas independently

Stage 1's dataset-server first-row endpoint failed for six configurations. This checks each configuration with the `datasets` streaming backend. Each result is reported independently; one configuration's schema is never substituted for another.

In [5]:
for configuration in ['moshaf_0.0', 'moshaf_5.0', 'moshaf_11.0', 'moshaf_19.0', 'moshaf_26.0', 'moshaf_30.0']:
    result = loader.inspect_schema(configuration)
    print(configuration, 'verified=', result.verified)
    if result.verified:
        print('field count:', len(result.features), '| fields:', list(result.features))
    else:
        print('error:', result.error)

moshaf_0.0 verified= True
field count: 24 | fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']


moshaf_5.0 verified= True
field count: 24 | fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']


moshaf_11.0 verified= True
field count: 24 | fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']


moshaf_19.0 verified= True
field count: 24 | fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']


moshaf_26.0 verified= True
field count: 24 | fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']


moshaf_30.0 verified= True
field count: 24 | fields: ['audio', 'segment_index', 'tarteel_transcript', 'moshaf_id', 'moshaf_name', 'reciter_id', 'reciter_arabic_name', 'reciter_english_name', 'sura_or_aya_index', 'index_type', 'sample_rate', 'duration_seconds', 'timestamp_seconds', 'imlaey', 'uthmani', 'has_quran', 'has_istiaatha', 'has_bismillah', 'has_sadaka', 'start_span', 'end_span', 'match_ratio', 'phonemes', 'sifat']


## 6. Iterate a few rows and handle access errors
Each selected configuration gets its own limit. The intentionally invalid config name below demonstrates explicit local rejection without trying another set.

In [6]:
for configured in loader.iter_configurations(['moshaf_0.1'], limit_per_configuration=2):
    print(configured.configuration, configured.record.get('segment_index'), configured.record.get('duration_seconds'))

try:
    loader.load_config('moshaf_not_a_real_configuration')
except Exception as exc:
    print('Access error reported:', type(exc).__name__, str(exc)[:240])

moshaf_0.1 001.0000 6.059999942779541
moshaf_0.1 001.0001 6.019999980926514
Access error reported: ValueError Unknown configuration 'moshaf_not_a_real_configuration'. Available configurations: moshaf_0.0, moshaf_0.1, moshaf_0.2, moshaf_0.3, moshaf_1.0, moshaf_11.0, moshaf_12.0, moshaf_13.0, moshaf_19.0, moshaf_2.0, moshaf_2.1, moshaf_22.0, moshaf_2


## Summary
This notebook accessed only a few rows from one recording set, its small metadata tables, and one decoded audio clip. It did not download the full dataset or alter audio or labels.